In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and run:\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test that the local Ollama server is running, use:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about running **Ollama** locally.\n\nThe closest related note says you can run the **course locally** instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and other needed tools, and that you should document your setup to keep it reproducible.\n\nIf you meant something else by “Olama,” let me know.'

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can join if it’s still open for enrollment.\n\nIf you want, I can help you figure out:\n- whether the course is still accepting students,\n- how to enroll,\n- or what to do if it already started.\n\nIf you share the course name or link, I can give more specific guidance.'

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enroll late registration"}', call_id='call_stAREfvz5NBePmRGdIZVQHPu', name='search', type='function_call', id='fc_098091a69f356822006a399792e930819d9788a303d10abe78', namespace=None, status='completed')]

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(776, 30)

In [20]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [22]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [23]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course enroll late registration can I join"}
function_call: search {"query":"course FAQ enrollment access late join new student"}
function_call: search {"query":"discovered the course can I join it FAQ"}


In [24]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [25]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install start server FAQ"}
function_call: search {"query":"run Ollama locally command line model pull serve"}
function_call: search {"query":"Ollama localhost FAQ local setup"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - macOS: download and install from https://ollama.com/download
   - Windows: download the `.msi` from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model if needed and runs it locally with a chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a JSON response.

4. **If the server isn’t running**
   You can start it explicitly:
   ```bash
   ollama serve
   ```
   If you’re in a notebook or background process, the course FAQ suggests:
   ```bash
   !nohup olla

'To run Ollama locally:\n\n1. **Install Ollama**\n   - macOS: download and install from https://ollama.com/download\n   - Windows: download the `.msi` from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model if needed and runs it locally with a chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a JSON response.\n\n4. **If the server isn’t running**\n   You can start it explicitly:\n   ```bash\n   ollama serve\n   ```\n   If you’re in a notebook or background process, the course FAQ suggests:\n   ```bash\n   !nohup ollama serve > nohup.out 2>&1 &\n   ```\n\n5. **Optional: use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "use

In [26]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late can I still join discovered the course enrollment late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important note: if you want a certificate, you need to submit your project while submissions are still being accepted.

If you’d like, I can also help with how to start the course now or explain the certificate requirements. Any other areas you want to explore?


'Yes — you can still join the course.\n\nOne important note: if you want a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you’d like, I can also help with how to start the course now or explain the certificate requirements. Any other areas you want to explore?'

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join access FAQ"}
iteration #2...
function_call: search {"query":"registration accepted can just start learning and submitting homework while the form is open registration just gauge interest confirmation email accepted FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open. Also, you can start learning and submitting homework even if you didn’t register in advance; registration is mainly to gauge interest.

If you want, I can also help you with how to get started or how the certificate works.


'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open. Also, you can start learning and submitting homework even if you didn’t register in advance; registration is mainly to gauge interest.\n\nIf you want, I can also help you with how to get started or how the certificate works.'

In [28]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [29]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [30]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [31]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [32]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)